# 02. Data Preprocessing


In [1]:
import pandas as pd
import numpy as np
import pathlib
import os

RAW_DATA_PATH = pathlib.Path("data/raw")
ADJUSTED_PATH = pathlib.Path("data/adjusted")
ADJUSTED_PATH.mkdir(parents=True, exist_ok=True)

csv_files = sorted(list(RAW_DATA_PATH.glob("*.csv")))

## 1. Defining Robust Parsing Logic
The SJPD dataset has inconsistent timestamp formats across years (e.g., some have 'PS' suffixes, others use different delimiters). We define custom functions to handle this complexity.

In [2]:
def parse_cdts_series(raw_series: pd.Series) -> pd.Series:
    # Normalize raw values and remove known suffix noise.
    s = (
        raw_series.astype("string")
        .str.upper()
        .str.replace("PS", "", regex=False)
        .str.strip()
    )

    # Primary parse: first 14 digits interpreted as YYYYMMDDHHMMSS.
    digits14 = s.str.extract(r"(\d{14})", expand=False)
    parsed = pd.to_datetime(digits14, format="%Y%m%d%H%M%S", errors="coerce")

    # Secondary parse: 12-digit timestamps (YYYYMMDDHHMM) -> append seconds.
    missing_mask = parsed.isna()
    if missing_mask.any():
        digits12 = s[missing_mask].str.extract(r"(\d{12})", expand=False)
        parsed12 = pd.to_datetime(digits12 + "00", format="%Y%m%d%H%M%S", errors="coerce")
        parsed.loc[missing_mask] = parsed12.values

    # Final fallback: let pandas infer any remaining odd formats.
    missing_mask = parsed.isna()
    if missing_mask.any():
        parsed_fallback = pd.to_datetime(s[missing_mask], errors="coerce")
        parsed.loc[missing_mask] = parsed_fallback.values

    return parsed


## 2. Processing All Files
We load each file, apply our cleaning logic, and flag important features like 'Canceled' calls and 'Priority 1' calls.

In [3]:
processed_dfs = []

for file_path in csv_files:
    print(f"Processing {file_path.name}...")
    df = pd.read_csv(file_path, low_memory=False)
    
    # Capture raw CDTS for traceability checks
    df["CDTS_RAW"] = df["CDTS"].astype("string")
    
    # Apply robust parsing
    df["CDTS"] = parse_cdts_series(df["CDTS"])
    
    # Simple flags for outcome tracking
    if "FINAL_DISPO" in df.columns:
        df["is_canceled"] = df["FINAL_DISPO"].str.contains("CAN", case=False, na=False)
    else:
        df["is_canceled"] = False
        
    if "PRIORITY" in df.columns:
        df["is_priority_1"] = df["PRIORITY"].astype(str).str.contains("1", na=False)
    else:
        df["is_priority_1"] = False
        
    processed_dfs.append(df)

master_df = pd.concat(processed_dfs, ignore_index=True)

Processing policecalls2016.csv...


Processing policecalls2017.csv...


Processing policecalls2018.csv...


Processing policecalls2019.csv...


Processing policecalls2020.csv...


...

Processing policecalls2022.csv...


Processing policecalls2023.csv...


Processing policecalls2024.csv...


Processing policecalls2025.csv...


Processing policecalls2026.csv...


## 3. Traceability Check
Let's compare the raw input vs the cleaned output to ensure our parsing logic didn't drop data.

In [4]:
display(master_df[["CDTS_RAW", "CDTS", "is_canceled", "is_priority_1"]].head())
print("Missing CDTS after parsing:", master_df["CDTS"].isna().sum())

,CDTS_RAW,CDTS,is_canceled,is_priority_1
0,20160514222222PD,2016-05-14 22:22:22,False,False
1,20160514214406PD,2016-05-14 21:44:06,False,False
2,20160514212618PD,2016-05-14 21:26:18,False,False
3,20160514225346PD,2016-05-14 22:53:46,False,False
4,20160514224627PD,2016-05-14 22:46:27,False,False


Missing CDTS after parsing: 0


## 4. Save Final Dataset
We save the combined file to the `adjusted` directory for use in downstream EDA and Modeling notebooks.

In [5]:
out_path = ADJUSTED_PATH / "policecalls_adjusted_all.csv"
master_df.to_csv(out_path, index=False)
print(f"Saved cleaned dataset to: {out_path}")

Saved cleaned dataset to: data/adjusted/policecalls_adjusted_all.csv
